
# Week 4 — Regression Report (EDA & Preprocessing Only)

**Dataset:** `pakwheels_used_cars.csv`  
**Objective:** Clean, analyze, visualize, and preprocess the data for a regression task (predicting `price`).  
**Note:** Per assignment, **no regression models are trained**—we stop at a clean, model-ready dataset.

## Checklist
- [x] Data loading & preview
- [x] Missing values handling (with justification)
- [x] Categorical encoding & numeric scaling
- [x] Train/test split (80/20)
- [x] EDA & Matplotlib visualizations (histograms, scatter, box, correlation heatmap)
- [x] Save preprocessed train/test arrays ready for modeling (optional)


In [ ]:

# Imports & Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

DATA_DIR = Path("..") / "data"
# Support either standardized filename or original upload
CSV_CANDIDATES = ["pakwheels_used_cars.csv", "pakwheels_used_car_data_v02.csv"]
for c in CSV_CANDIDATES:
    p = DATA_DIR / c
    if p.exists():
        CSV_PATH = p
        break
else:
    raise FileNotFoundError("PakWheels dataset not found in ../data")

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
df.head()



## 1) Data Cleaning & Preparation

### 1.1 Target & Features
- **Target:** `price` (PKR)
- **Features:** All other columns (mixed numeric/categorical)

### 1.2 Missing Values
- **Numeric:** median imputation (robust to outliers like high mileage or engine_cc).
- **Categorical:** most-frequent (mode) imputation to keep prevalent categories.


In [ ]:

display(df.info())
na_counts = df.isna().sum().sort_values(ascending=False)
na_counts


In [ ]:

target_col = "price"
assert target_col in df.columns, "Target 'price' not found!"

X = df.drop(columns=[target_col])
y = df[target_col]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_cols, categorical_cols, y.describe()


In [ ]:

# 1.3 Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

len(X_train), len(X_test)



## 2) Exploratory Data Analysis (EDA) & Visualization

We focus on price distributions, relationships with key numeric predictors (e.g., `mileage`, `engine_cc`, `year`), outliers, and correlations.


In [ ]:

# Summary statistics
display(df.describe(include='all'))

# Histograms for numeric features
for col in numeric_cols:
    plt.figure()
    plt.hist(df[col].dropna(), bins=30)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

# Box plots for numeric features
for col in numeric_cols:
    plt.figure()
    plt.boxplot(df[col].dropna(), vert=True)
    plt.title(f"Box plot of {col}")
    plt.ylabel(col)
    plt.show()

# Scatter plots: a few pairs with price
pairs = [c for c in numeric_cols if c != "price"][:4]
for xcol in pairs:
    plt.figure()
    plt.scatter(df[xcol], y)
    plt.title(f"Scatter: {xcol} vs price")
    plt.xlabel(xcol)
    plt.ylabel("price")
    plt.show()

# Correlation heatmap (numeric)
num_cols_for_corr = [c for c in df.select_dtypes(include=[np.number]).columns if c != "price"]
if len(num_cols_for_corr) > 1:
    corr = df[num_cols_for_corr + ["price"]].corr(numeric_only=True)
    plt.figure(figsize=(8,6))
    plt.imshow(corr, interpolation='nearest')
    plt.title("Correlation Heatmap (numeric incl. price)")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.colorbar()
    plt.tight_layout()
    plt.show()



## 3) Preprocessing Pipeline (Model-Ready)

We build a scikit-learn `ColumnTransformer` that:
- imputes numeric with median and scales with `StandardScaler`
- imputes categorical with most-frequent and encodes with `OneHotEncoder(handle_unknown="ignore")`

We then **fit** the preprocessing on the training data and **transform** both train and test splits to obtain model-ready arrays for any regression model (not trained here as per instructions).


In [ ]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

# Fit on training, transform both
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Train transformed shape:", X_train_ready.shape)
print("Test transformed shape:", X_test_ready.shape)

# (Optional) Save preprocessed arrays for downstream modeling
import joblib
joblib.dump(preprocessor, "preprocessor_pakwheels.joblib")



## 4) Conclusion & Next Steps

- The dataset has been cleaned, explored, and preprocessed into a model-ready form.
- To proceed, you can plug `X_train_ready`, `y_train` into any regressor (e.g., Linear Regression, Random Forest Regressor, XGBoost) and evaluate on `X_test_ready`, `y_test`.
- Consider log-transforming `price` if right-skewed, adding interaction terms, or domain-specific features (e.g., car age = current_year - year).

**Submission Tip:** Commit this notebook, the `../data` folder, and any saved artifacts (e.g., `preprocessor_pakwheels.joblib`) to your GitHub under `week_4/regression`.
